In [ ]:
# =========================================================
# COMPLETE MACHINE LEARNING PIPELINE
# =========================================================
# Features:
# - CSV input
# - Stratified K-Fold Cross Validation
# - Baseline Models
# - Advanced Models
# - Stacking Ensemble
# - MLflow Tracking
# - Model Saving & Loading
# - Evaluation Metrics:
#     Accuracy
#     Precision
#     Recall
#     F1-Score
#     ROC-AUC
# =========================================================

# =========================================================
# INSTALL REQUIRED PACKAGES
# =========================================================
# pip install pandas numpy scikit-learn mlflow xgboost lightgbm

# =========================================================
# IMPORTS
# =========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# =========================================================
# BASELINE MODELS
# =========================================================

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    StackingClassifier
)

# =========================================================
# ADVANCED MODELS
# =========================================================

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# =========================================================
# META MODEL FOR STACKING
# =========================================================

from sklearn.linear_model import LogisticRegression as MetaModel

# =========================================================
# CONFIGURATION
# =========================================================

CSV_PATH = "../datasets/cleaned_train.csv"

TARGET_COLUMN = "y"

RANDOM_STATE = 42

N_SPLITS = 5

PRIMARY_METRIC = "f1"

# =========================================================
# LOAD DATA
# =========================================================

print("\nLoading Dataset...")

df = pd.read_csv(CSV_PATH)

print(f"Dataset Shape: {df.shape}")

# =========================================================
# SEPARATE FEATURES AND TARGET
# =========================================================

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

# =========================================================
# ENCODE TARGET IF CATEGORICAL
# =========================================================

if y.dtype == "object":

    le = LabelEncoder()
    y = le.fit_transform(y)

# =========================================================
# IDENTIFY COLUMN TYPES
# =========================================================

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns

# =========================================================
# PREPROCESSING PIPELINE
# =========================================================

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# =========================================================
# STRATIFIED K-FOLD
# =========================================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# =========================================================
# DEFINE MODELS
# =========================================================

baseline_models = {

    "LogisticRegression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),

    "RandomForest": Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=RANDOM_STATE
        ))
    ])
}

advanced_models = {

    "XGBoost": Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            eval_metric="logloss",
            random_state=RANDOM_STATE
        ))
    ]),

    "LightGBM": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LGBMClassifier(
            random_state=RANDOM_STATE
        ))
    ]),

    "GradientBoosting": Pipeline([
        ("preprocessor", preprocessor),
        ("model", GradientBoostingClassifier(
            random_state=RANDOM_STATE
        ))
    ]),

    "SVM": Pipeline([
        ("preprocessor", preprocessor),
        ("model", SVC(
            probability=True,
            random_state=RANDOM_STATE
        ))
    ]),

    "NaiveBayes": Pipeline([
        ("preprocessor", preprocessor),
        ("model", GaussianNB())
    ]),

    "DeepLearning": Pipeline([
        ("preprocessor", preprocessor),
        ("model", MLPClassifier(
            hidden_layer_sizes=(128, 64),
            max_iter=500,
            random_state=RANDOM_STATE
        ))
    ])
}

# =========================================================
# EVALUATION METRICS
# =========================================================

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_weighted",
    "recall": "recall_weighted",
    "f1": "f1_weighted",
    "roc_auc": "roc_auc_ovr"
}

# =========================================================
# MODEL TRAINING FUNCTION
# =========================================================

def evaluate_models(models, category):

    best_model = None
    best_name = None
    best_score = 0

    for name, model in models.items():

        print("\n" + "="*60)
        print(f"Training Model: {name}")
        print("="*60)

        with mlflow.start_run(run_name=name):

            # ---------------------------------------------
            # CROSS VALIDATION
            # ---------------------------------------------

            cv_results = cross_validate(
                model,
                X,
                y,
                cv=skf,
                scoring=scoring,
                return_train_score=False
            )

            # ---------------------------------------------
            # CALCULATE METRICS
            # ---------------------------------------------

            accuracy = np.mean(cv_results["test_accuracy"])
            precision = np.mean(cv_results["test_precision"])
            recall = np.mean(cv_results["test_recall"])
            f1 = np.mean(cv_results["test_f1"])
            roc_auc = np.mean(cv_results["test_roc_auc"])

            # ---------------------------------------------
            # PRINT RESULTS
            # ---------------------------------------------

            print(f"Accuracy : {accuracy:.4f}")
            print(f"Precision: {precision:.4f}")
            print(f"Recall   : {recall:.4f}")
            print(f"F1 Score : {f1:.4f}")
            print(f"ROC-AUC  : {roc_auc:.4f}")

            # ---------------------------------------------
            # LOG PARAMETERS
            # ---------------------------------------------

            mlflow.log_param("model_name", name)
            mlflow.log_param("category", category)

            # ---------------------------------------------
            # LOG METRICS
            # ---------------------------------------------

            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("roc_auc", roc_auc)

            # ---------------------------------------------
            # TRAIN FULL MODEL
            # ---------------------------------------------

            model.fit(X, y)

            # ---------------------------------------------
            # SAVE MODEL
            # ---------------------------------------------

            mlflow.sklearn.log_model(
                sk_model=model,
                artifact_path=name
            )

            # ---------------------------------------------
            # PREDICTIONS
            # ---------------------------------------------

            predictions = model.predict(X)

            # ---------------------------------------------
            # CLASSIFICATION REPORT
            # ---------------------------------------------

            report = classification_report(
                y,
                predictions
            )

            report_file = f"{name}_classification_report.txt"

            with open(report_file, "w") as f:
                f.write(report)

            mlflow.log_artifact(report_file)

            # ---------------------------------------------
            # CONFUSION MATRIX
            # ---------------------------------------------

            cm = confusion_matrix(y, predictions)

            cm_file = f"{name}_confusion_matrix.csv"

            pd.DataFrame(cm).to_csv(
                cm_file,
                index=False
            )

            mlflow.log_artifact(cm_file)

            # ---------------------------------------------
            # SELECT BEST MODEL
            # ---------------------------------------------

            if f1 > best_score:

                best_score = f1
                best_model = model
                best_name = name

    return best_name, best_model, best_score

# =========================================================
# TRAIN BASELINE MODELS
# =========================================================

print("\n")
print("#"*70)
print("TRAINING BASELINE MODELS")
print("#"*70)

best_baseline_name, best_baseline_model, baseline_score = evaluate_models(
    baseline_models,
    "baseline"
)

print("\n")
print("="*70)
print(f"BEST BASELINE MODEL: {best_baseline_name}")
print(f"BEST F1 SCORE      : {baseline_score:.4f}")
print("="*70)

# =========================================================
# TRAIN ADVANCED MODELS
# =========================================================

print("\n")
print("#"*70)
print("TRAINING ADVANCED MODELS")
print("#"*70)

best_advanced_name, best_advanced_model, advanced_score = evaluate_models(
    advanced_models,
    "advanced"
)

print("\n")
print("="*70)
print(f"BEST ADVANCED MODEL: {best_advanced_name}")
print(f"BEST F1 SCORE      : {advanced_score:.4f}")
print("="*70)

# =========================================================
# STACKING ENSEMBLE
# =========================================================

print("\n")
print("#"*70)
print("TRAINING STACKING ENSEMBLE")
print("#"*70)

estimators = [
    ("baseline", best_baseline_model),
    ("advanced", best_advanced_model)
]

stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=MetaModel(),
    cv=5
)

with mlflow.start_run(run_name="Stacking_Ensemble"):

    # -----------------------------------------------------
    # CROSS VALIDATION
    # -----------------------------------------------------

    ensemble_results = cross_validate(
        stacking_model,
        X,
        y,
        cv=skf,
        scoring=scoring
    )

    # -----------------------------------------------------
    # METRICS
    # -----------------------------------------------------

    ensemble_accuracy = np.mean(
        ensemble_results["test_accuracy"]
    )

    ensemble_precision = np.mean(
        ensemble_results["test_precision"]
    )

    ensemble_recall = np.mean(
        ensemble_results["test_recall"]
    )

    ensemble_f1 = np.mean(
        ensemble_results["test_f1"]
    )

    ensemble_roc_auc = np.mean(
        ensemble_results["test_roc_auc"]
    )

    # -----------------------------------------------------
    # PRINT RESULTS
    # -----------------------------------------------------

    print(f"Accuracy : {ensemble_accuracy:.4f}")
    print(f"Precision: {ensemble_precision:.4f}")
    print(f"Recall   : {ensemble_recall:.4f}")
    print(f"F1 Score : {ensemble_f1:.4f}")
    print(f"ROC-AUC  : {ensemble_roc_auc:.4f}")

    # -----------------------------------------------------
    # LOG METRICS
    # -----------------------------------------------------

    mlflow.log_metric("accuracy", ensemble_accuracy)
    mlflow.log_metric("precision", ensemble_precision)
    mlflow.log_metric("recall", ensemble_recall)
    mlflow.log_metric("f1_score", ensemble_f1)
    mlflow.log_metric("roc_auc", ensemble_roc_auc)

    # -----------------------------------------------------
    # TRAIN FINAL ENSEMBLE
    # -----------------------------------------------------

    stacking_model.fit(X, y)

    # -----------------------------------------------------
    # SAVE ENSEMBLE MODEL
    # -----------------------------------------------------

    mlflow.sklearn.log_model(
        stacking_model,
        "stacking_ensemble"
    )

    # -----------------------------------------------------
    # FINAL PREDICTIONS
    # -----------------------------------------------------

    final_predictions = stacking_model.predict(X)

    # -----------------------------------------------------
    # SAVE CLASSIFICATION REPORT
    # -----------------------------------------------------

    final_report = classification_report(
        y,
        final_predictions
    )

    with open("stacking_classification_report.txt", "w") as f:
        f.write(final_report)

    mlflow.log_artifact(
        "stacking_classification_report.txt"
    )

# =========================================================
# FINAL RESULTS
# =========================================================

print("\n")
print("="*70)
print("FINAL RESULTS")
print("="*70)

print(f"Best Baseline Model : {best_baseline_name}")
print(f"Best Advanced Model : {best_advanced_name}")

print("\nFinal Ensemble Metrics")

print(f"Accuracy : {ensemble_accuracy:.4f}")
print(f"Precision: {ensemble_precision:.4f}")
print(f"Recall   : {ensemble_recall:.4f}")
print(f"F1 Score : {ensemble_f1:.4f}")
print(f"ROC-AUC  : {ensemble_roc_auc:.4f}")

print("\n")
print("="*70)
print("TRAINING COMPLETED SUCCESSFULLY")
print("="*70)

# =========================================================
# HOW TO RUN
# =========================================================
# 1. Put your CSV file inside:
#       data/dataset.csv
#
# 2. Update:
#       TARGET_COLUMN = "your_target_column"
#
# 3. Run:
#       python main.py
#
# 4. Start MLflow UI:
#       mlflow ui
#
# 5. Open:
#       http://127.0.0.1:5000
#
# =========================================================
# HOW TO LOAD SAVED MODEL
# =========================================================
#
# import mlflow.sklearn
#
# model = mlflow.sklearn.load_model(
#     "runs:/RUN_ID/stacking_ensemble"
# )
#
# predictions = model.predict(X_test)
#
# =========================================================

In [ ]:
import os
import json
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# ----------------------------
# CONFIG
# ----------------------------

config = {}
with open("../utils/config.json", "r") as f:
    config = json.load(f)

if config:
        TARGET_COL = config["target_columns"]
        model_types = config.get("model_types", ["base_models", "advanced_models"])
        drop_cols = config.get("target_columns", [])
        models = config.get("models", {})
        

# ----------------------------
# LOAD DATA
# ----------------------------
df = pd.read_csv("../datasets/cleaned_train.csv")  # replace with your CSV file path

x = df.drop(columns=drop_cols)
y = df[TARGET_COL]

# ----------------------------
# MODELS
# ----------------------------
all_models = {
            "base_models": {
                                "logisticregression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
                                "randomforest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
                            },
            "advanced_models": {
                                    "XGBoost": XGBClassifier(
                                                                n_estimators=100,
                                                                max_depth=6,
                                                                learning_rate=0.05,
                                                                objective="binary:logistic",
                                                                eval_metric="logloss"
                                                            ),
                                    "light_gbm": LGBMClassifier(
                                                                    n_estimators=200,
                                                                    learning_rate=0.05,
                                                                    max_depth=-1
                                                                ),
                                    # "GradientBoosting": GradientBoostingClassifier(),
                                    # "svm": LinearSVC(),
                                    # "naive_bayes": GaussianNB(),
                                    # "DeepLearning": Sequential([
                                    #                                 Dense(64, activation='relu', input_shape=(X.shape[1],)),
                                    #                                 Dense(32, activation='relu'),
                                    #                                 Dense(1, activation='sigmoid')  # binary output (0/1)
                                    #                             ])
                                }
    }

# for deep learning, we need to compile and train separately
# # Compile
# dl_model.compile(
#     optimizer='adam',
#     loss='binary_crossentropy',
#     metrics=['accuracy']
# )

# # Train
# dl_model.fit(
#     X_train, y_train,
#     epochs=10,
#     batch_size=256,
#     validation_split=0.2,
#     verbose=1
# )

# ----------------------------
# EVALUATION STORAGE
# ----------------------------


# ----------------------------
# MLflow Setup
# ----------------------------

for model_type, models in all_models.items():
    MODEL_DIR = os.path.join(
                config["ml_model_dirs"]["base_dir"],
                config["ml_model_dirs"][model_type]
            )
    model_metrics_path = os.path.join(MODEL_DIR, "model_metrics.json")
    results = {
                    "models": {model_type: {}}
                }

    if os.path.exists(os.path.join(MODEL_DIR, "model_metrics.json")):
        with open(os.path.join(MODEL_DIR, "model_metrics.json"), "r") as f:
            existing_results = json.load(f)
        results = results | existing_results  # Merge with existing results
    best_model_name = None
    best_score = -1
    mlruns_path = os.path.abspath(os.path.join(MODEL_DIR, "mlruns"))
    os.makedirs(mlruns_path, exist_ok=True)
    mlflow.set_tracking_uri(f"file:///{mlruns_path.replace(os.sep, '/')}")
    mlflow.set_experiment(model_type)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for name, model in models.items():
        print("===========================================================================")
        print(f"Training {name} with Stratified K-Fold CV...")

        fold_metrics = {
                            "accuracy": [],
                            "precision": [],
                            "recall": [],
                            "f1": [],
                            "roc_auc": []
                        }

        mlflow.end_run()  # safety reset
        with mlflow.start_run(run_name=name) as run:
            run_id = run.info.run_id
            for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

                X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
                y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

                model.fit(X_train, y_train)
                y_pred = model.predict(X_val)

                # ROC-AUC (binary safe)
                if hasattr(model, "predict_proba"):
                    y_prob = model.predict_proba(X_val)[:, 1]
                    roc_auc = roc_auc_score(y_val, y_prob)
                else:
                    roc_auc = 0.0

                acc = accuracy_score(y_val, y_pred)
                prec = precision_score(y_val, y_pred, zero_division=0)
                rec = recall_score(y_val, y_pred, zero_division=0)
                f1 = f1_score(y_val, y_pred, zero_division=0)

                fold_metrics["accuracy"].append(acc)
                fold_metrics["precision"].append(prec)
                fold_metrics["recall"].append(rec)
                fold_metrics["f1"].append(f1)
                fold_metrics["roc_auc"].append(roc_auc)

            # ----------------------------
            # AVG ACROSS FOLDS
            # ----------------------------
            accuracy = np.mean(fold_metrics["accuracy"])
            precision = np.mean(fold_metrics["precision"])
            recall = np.mean(fold_metrics["recall"])
            f1 = np.mean(fold_metrics["f1"])
            roc_auc = np.mean(fold_metrics["roc_auc"])

            # ----------------------------
            # LOG TO MLflow
            # ----------------------------
            mlflow.log_metric(f"{name}_accuracy", accuracy)
            mlflow.log_metric(f"{name}_precision", precision)
            mlflow.log_metric(f"{name}_recall", recall)
            mlflow.log_metric(f"{name}_f1_score", f1)
            mlflow.log_metric(f"{name}_roc_auc", roc_auc)

            # ----------------------------
            # FINAL TRAIN (FULL DATA)
            # ----------------------------
            model.fit(X, y)
            mlflow.sklearn.log_model(model, name)

            # ----------------------------
            # CUSTOM SCORE
            # ----------------------------
            score = (
                        0.4 * precision +
                        0.3 * roc_auc +
                        0.2 * f1 +
                        0.1 * accuracy
                    )

            print(f"{name} CV Score: {score:.4f}")

            # store results
            results["models"][model_type][name] = {
                "accuracy": float(accuracy),
                "precision": float(precision),
                "recall": float(recall),
                "f1-score": float(f1),
                "roc-auc": float(roc_auc),
                "bestmodel": "no"
            }

            # best model tracking
            if score > best_score:
                best_score = score
                best_model_name = name

        print("===========================================================================")
        print("\n")

    # mark best model
    if best_model_name:
        results["models"][model_type][best_model_name]["bestmodel"] = "yes"

    # save JSON
    with open(model_metrics_path, "w") as f:
        json.dump(results, f, indent=4)

    print("Training complete.")
    print("Best model:", best_model_name)

2026/05/17 11:20:43 INFO mlflow.tracking.fluent: Experiment with name 'base_models' does not exist. Creating a new experiment.


Training logisticregression with Stratified K-Fold CV...


2026/05/17 11:21:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 11:21:25 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


logisticregression CV Score: 0.7654


Training randomforest with Stratified K-Fold CV...


2026/05/17 11:28:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 11:28:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/05/17 11:28:44 INFO mlflow.tracking.fluent: Experiment with name 'advanced_models' does not exist. Creating a new experiment.


randomforest CV Score: 0.8069


Training complete.
Best model: randomforest
Training XGBoost with Stratified K-Fold CV...


2026/05/17 11:29:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 11:29:17 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


XGBoost CV Score: 0.8148


Training light_gbm with Stratified K-Fold CV...
[LightGBM] [Info] Number of positive: 72391, number of negative: 527609
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017942 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1538
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.120652 -> initscore=-1.986273
[LightGBM] [Info] Start training from score -1.986273
[LightGBM] [Info] Number of positive: 72391, number of negative: 527609
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020551 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1538
[LightGBM] [Info] Number of data 

2026/05/17 11:29:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 11:30:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


light_gbm CV Score: 0.8290


Training complete.
Best model: light_gbm
